MIT License

Copyright (c) 2026 THALES

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Cost Model Comparison & Weight Tuning

This notebook compares PASSAGE cost models using `grid:astar` and tunes their
weight parameter by balancing three objectives:

1. **Maximize path diversity** — Hausdorff distance from the flat `distance` baseline
2. **Minimize boundary artifacts** — penalize paths that hug image edges (both overall fraction and contiguous runs)
3. **Minimize weight at asymptote** — prefer the smallest weight achieving near-peak score

**Combined score** per (cost_model, weight):

$$\text{score}(w) = \hat{H}(w) \cdot \bigl(1 - B(w)\bigr) \cdot P_{\text{run}}(w)$$

where $\hat{H}(w) \in [0,1]$ is the normalized mean Hausdorff distance from the
`distance` baseline, $B(w) \in [0,1]$ is the mean fraction of path points
within a margin of the image boundary, and $P_{\text{run}}(w) \in [0,1]$ is a
hard penalty for contiguous boundary runs:

$$P_{\text{run}}(w) = 1 - \text{clamp}\!\left(\frac{C(w) - \theta}{\theta},\; 0,\; 1\right)$$

where $C(w)$ is the mean max-contiguous-boundary fraction and $\theta$ is the
configurable threshold (`MAX_BOUNDARY_RUN_FRAC`, default 3%). Weights where
$C(w) \ge 2\theta$ receive a score of zero.

Among weights with $\text{score} \ge \tau \cdot \max(\text{score})$, we pick the
**smallest** weight (asymptote minimization).

In [ ]:
import importlib
import json
import logging
import math
import multiprocessing as mp
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loguru import logger as loguru_logger
from matplotlib.lines import Line2D
from tqdm import tqdm

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

# Reload to pick up new functions added to pathfinding_utils
import passage.pathfinding_utils as pathfinding_utils  # noqa: E402

importlib.reload(pathfinding_utils)

from passage.generate import find_path, load_tile, _assemble_crop_from_tiles  # noqa: E402
from passage.pathfinding_utils import (  # noqa: E402
    COST_MODELS,
    boundary_fraction,
    hausdorff_distance,
    max_contiguous_boundary_fraction,
)

In [ ]:
# Parameters (papermill)
RESOLUTION = 256
NUM_SAMPLES = 1000
NUM_DISPLAY_SAMPLES = 20
BOUNDARY_MARGIN = 2  # pixels from edge considered "near boundary"
SCORE_THRESHOLD = 0.95  # fraction of max score for asymptote selection
MAX_BOUNDARY_RUN_FRAC = 0.03  # max allowed contiguous boundary run as fraction of path length
NUM_WORKERS = min(mp.cpu_count(), 16)  # parallel workers for sample generation

In [ ]:
# Configuration
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"

WEIGHTS = np.pow(10, np.linspace(0, 7, num=40)).tolist()
COST_MODELS_LIST = sorted(list(COST_MODELS))
COST_MODELS_TUNABLE = [cm for cm in COST_MODELS_LIST if cm != "distance"]
TILE_SIZE_PX = 3600

# Color palette for cost models (consistent across all plots)
CM_COLORS = {cm: plt.cm.Set2(i / max(1, len(COST_MODELS_LIST) - 1)) for i, cm in enumerate(COST_MODELS_LIST)}

print(f"Resolution: {RESOLUTION}")
print(f"Num samples: {NUM_SAMPLES}")
print(f"Num display samples: {NUM_DISPLAY_SAMPLES}")
print(f"Weight range: [{WEIGHTS[0]:.1f} ... {WEIGHTS[-1]:.1f}] ({len(WEIGHTS)} values)")
print(f"Cost models: {COST_MODELS_LIST}")
print(f"Tunable (non-distance): {COST_MODELS_TUNABLE}")

In [ ]:
# Load calibration data
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)

global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")
print(f"Number of tiles: {len(tile_stats_df)}")

## Helper Functions

In [ ]:
def generate_random_crop(
    resolution: int,
    tile_stats_df: pd.DataFrame,
    tiles_dirpath: Path,
    global_min: float,
    global_max: float,
    seed: int | None = None,
) -> np.ndarray | None:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    half_res = resolution // 2
    half_tiles = max(1, math.ceil(half_res / TILE_SIZE_PX))
    n_tiles = 2 * half_tiles + 1
    grid_h = n_tiles * TILE_SIZE_PX
    grid_w = n_tiles * TILE_SIZE_PX

    for _ in range(10):
        target_elev = random.uniform(global_min, global_max)
        candidates = tile_stats_df[(tile_stats_df["min"] <= target_elev) & (tile_stats_df["max"] >= target_elev)]
        if candidates.empty:
            continue
        selected = candidates.sample(n=1).iloc[0]
        center_lat = int(selected["lat"])
        center_lon = int(selected["lon"])
        if grid_h < resolution or grid_w < resolution:
            continue
        y1 = random.randint(0, grid_h - resolution)
        x1 = random.randint(0, grid_w - resolution)
        center_tile = load_tile(tiles_dirpath, center_lat, center_lon)
        crop = _assemble_crop_from_tiles(
            tiles_dirpath=tiles_dirpath,
            center_lat=center_lat,
            center_lon=center_lon,
            half_tiles=half_tiles,
            y1=y1,
            x1=x1,
            resolution=resolution,
            center_tile=center_tile,
        )
        return crop
    return None


def get_marker_positions(resolution: int) -> tuple[tuple[int, int], tuple[int, int]]:
    pos1 = int(0.2 * resolution)
    pos2 = int(0.8 * resolution)
    return (pos1, pos1), (pos2, pos2)


def compute_path_for_crop(
    crop: np.ndarray,
    start: tuple[int, int],
    end: tuple[int, int],
    cost_model_name: str,
    weight: float,
) -> list:
    solver_config = {
        "name": "grid:astar",
        "diagonal_movement": "always",
        "weight": float(weight),
        "cost_model": cost_model_name,
        "heuristic": "euclidean",
    }
    return find_path(crop, start, end, solver_config)


def _process_one_sample(i):
    """Worker: generate one sample and compute all paths (for multiprocessing)."""
    crop = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=1000 + i)
    if crop is None:
        return None
    start, end = get_marker_positions(RESOLUTION)
    paths = {}
    for cm in COST_MODELS_LIST:
        paths[cm] = {}
        for w in WEIGHTS:
            paths[cm][w] = compute_path_for_crop(crop, start, end, cm, w)
    return {"index": i + 1, "crop": crop, "start": start, "end": end, "paths": paths}


def _process_one_display_sample(i):
    """Worker: generate one display sample with tuned weights (for multiprocessing)."""
    crop = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=5000 + i)
    if crop is None:
        return None
    start, end = get_marker_positions(RESOLUTION)
    paths = {}
    paths["distance"] = compute_path_for_crop(crop, start, end, "distance", 1.0)
    for cm, w in weights_tuned.items():
        paths[cm] = compute_path_for_crop(crop, start, end, cm, w)
    return {"index": i + 1, "crop": crop, "start": start, "end": end, "paths": paths}

---

## 1. Generate Samples & Compute Paths

For each sample, compute paths with every (cost_model, weight) combination.

In [ ]:
t0 = time.time()
ctx = mp.get_context("fork")
with ctx.Pool(NUM_WORKERS) as pool:
    results = list(
        tqdm(
            pool.imap_unordered(_process_one_sample, range(NUM_SAMPLES)),
            total=NUM_SAMPLES,
            desc=f"Generating samples ({NUM_WORKERS} workers)",
        )
    )

samples = sorted([r for r in results if r is not None], key=lambda s: s["index"])

elapsed = time.time() - t0
print(f"\nGenerated {len(samples)}/{NUM_SAMPLES} samples in {elapsed:.1f}s ({NUM_WORKERS} workers)")

---

## 2. Path Length Analysis

How does path length vary with weight for each cost model?

In [ ]:
# Build per-(sample, cost_model, weight) records
records = []
for sample in samples:
    for cm in COST_MODELS_LIST:
        for w in WEIGHTS:
            path = sample["paths"][cm][w]
            records.append(
                {
                    "sample": sample["index"],
                    "cost_model": cm,
                    "weight": w,
                    "path_length": len(path),
                    "found": len(path) > 0,
                }
            )

results_df = pd.DataFrame(records)
print(f"Records: {len(results_df)}")
results_df.head()

In [ ]:
# Mean path length vs weight per cost model
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: mean path length
ax = axes[0]
for cm in COST_MODELS_LIST:
    sub = results_df[results_df["cost_model"] == cm]
    agg = sub.groupby("weight")["path_length"].mean()
    ax.plot(agg.index, agg.values, "o-", lw=1.2, markersize=3, label=cm, color=CM_COLORS[cm])
ax.set_xscale("log")
ax.set_xlabel("Weight")
ax.set_ylabel("Mean path length (px)")
ax.set_title("Mean Path Length vs Weight")
ax.legend(fontsize=8)
ax.grid(True, ls=":", alpha=0.5)

# Right: path-found rate
ax = axes[1]
for cm in COST_MODELS_LIST:
    sub = results_df[results_df["cost_model"] == cm]
    agg = sub.groupby("weight")["found"].mean()
    ax.plot(agg.index, agg.values, "o-", lw=1.2, markersize=3, label=cm, color=CM_COLORS[cm])
ax.set_xscale("log")
ax.set_xlabel("Weight")
ax.set_ylabel("Path found rate")
ax.set_title("Path Success Rate vs Weight")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8)
ax.grid(True, ls=":", alpha=0.5)

plt.tight_layout()
plt.show()

### Analysis: Path Length & Success Rate

**Left panel — Mean path length vs weight:**
- The `distance` model (flat cost) produces a constant, short path regardless of weight — it always follows the Euclidean-shortest route, confirming it as our "straight-line" baseline.
- All elevation-aware cost models show increasing path length with weight: higher weight amplifies the terrain penalty, forcing the solver to route around obstacles instead of crossing them.
- `elevation` and `slope_uphill` show the steepest length increase, indicating they generate the most tortuous detours. `absolute_elevation` saturates earliest, meaning its terrain signal is relatively weak compared to distance costs at moderate weights.
- Beyond ~$10^5$, most models plateau — the path cannot deviate much further within a 256×256 crop.

**Right panel — Path success rate:**
- All cost models maintain a 100% success rate across the full weight range. This confirms that the A* solver with 8-connectivity always finds a valid path within the crop, and no weight causes the solver to fail. This is expected since the grid is fully connected (diagonal movement = "always") and the start/end points are always at fixed positions (20%/80% of resolution).

---

## 3. Hausdorff Distance from `distance` Baseline

Measure how different each cost model's path is from the flat `distance` model
(straight-line baseline). Higher Hausdorff = more elevation-aware deviation.

In [ ]:
# Compute per-(sample, cost_model, weight): Hausdorff vs distance, boundary fraction, contiguous run
metric_records = []

for sample in tqdm(samples, desc="Computing metrics"):
    crop = sample["crop"]
    paths = sample["paths"]
    res = crop.shape[0]

    # distance model is weight-independent — use first weight as baseline
    distance_path = paths["distance"][WEIGHTS[0]]

    for cm in COST_MODELS_LIST:
        for w in WEIGHTS:
            path = paths[cm][w]
            hd = hausdorff_distance(path, distance_path)
            bf = boundary_fraction(path, res, BOUNDARY_MARGIN)
            mcbf = max_contiguous_boundary_fraction(path, res, BOUNDARY_MARGIN)
            metric_records.append(
                {
                    "sample": sample["index"],
                    "cost_model": cm,
                    "weight": w,
                    "hausdorff_vs_distance": hd,
                    "boundary_frac": bf,
                    "hits_boundary": bf > 0,
                    "max_contiguous_boundary_frac": mcbf,
                    "path_length": len(path),
                }
            )

metric_df = pd.DataFrame(metric_records)
print(f"Metric records: {len(metric_df)}")

In [ ]:
# Aggregate across samples
agg_df = (
    metric_df.groupby(["cost_model", "weight"])
    .agg(
        mean_hausdorff=("hausdorff_vs_distance", "mean"),
        std_hausdorff=("hausdorff_vs_distance", "std"),
        mean_boundary_frac=("boundary_frac", "mean"),
        boundary_hit_rate=("hits_boundary", "mean"),
        mean_max_contig_boundary=("max_contiguous_boundary_frac", "mean"),
        mean_path_length=("path_length", "mean"),
    )
    .reset_index()
)
agg_df.head()

In [ ]:
# Hausdorff distance vs weight — all tunable cost models overlaid
fig, ax = plt.subplots(figsize=(12, 6))

for cm in COST_MODELS_TUNABLE:
    sub = agg_df[agg_df["cost_model"] == cm].sort_values("weight")
    ax.plot(
        sub["weight"],
        sub["mean_hausdorff"],
        "o-",
        lw=1.5,
        markersize=4,
        label=cm,
        color=CM_COLORS[cm],
    )
    ax.fill_between(
        sub["weight"],
        sub["mean_hausdorff"] - sub["std_hausdorff"],
        sub["mean_hausdorff"] + sub["std_hausdorff"],
        alpha=0.15,
        color=CM_COLORS[cm],
    )

ax.set_xscale("log")
ax.set_xlabel("Weight")
ax.set_ylabel("Mean Hausdorff distance from `distance` baseline (px)")
ax.set_title("Path Diversity: Hausdorff Distance vs Weight")
ax.legend(fontsize=9)
ax.grid(True, ls=":", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Per-cost-model Hausdorff subplots with individual y-scales
n_cm = len(COST_MODELS_TUNABLE)
fig, axes = plt.subplots(1, n_cm, figsize=(5 * n_cm, 4), sharey=False)
if n_cm == 1:
    axes = [axes]

for idx, cm in enumerate(COST_MODELS_TUNABLE):
    ax = axes[idx]
    sub = agg_df[agg_df["cost_model"] == cm].sort_values("weight")
    ax.plot(sub["weight"], sub["mean_hausdorff"], "o-", lw=1.5, markersize=4, color=CM_COLORS[cm])
    ax.fill_between(
        sub["weight"],
        sub["mean_hausdorff"] - sub["std_hausdorff"],
        sub["mean_hausdorff"] + sub["std_hausdorff"],
        alpha=0.2,
        color=CM_COLORS[cm],
    )
    # Mark peak
    peak_idx = sub["mean_hausdorff"].idxmax()
    peak_row = sub.loc[peak_idx]
    ax.scatter([peak_row["weight"]], [peak_row["mean_hausdorff"]], color="red", s=50, zorder=10)
    ax.annotate(
        f"w={peak_row['weight']:.0f}",
        xy=(peak_row["weight"], peak_row["mean_hausdorff"]),
        xytext=(5, 8),
        textcoords="offset points",
        fontsize=8,
        color="red",
    )
    ax.set_xscale("log")
    ax.set_title(cm, fontsize=11)
    ax.set_xlabel("Weight")
    if idx == 0:
        ax.set_ylabel("Mean Hausdorff (px)")
    ax.grid(True, ls=":", alpha=0.5)

fig.suptitle("Hausdorff Distance per Cost Model", fontsize=13)
plt.tight_layout()
plt.show()

### Analysis: Hausdorff Distance from Distance Baseline

**Overlay plot:** All tunable cost models show a characteristic rise-then-plateau pattern in Hausdorff distance vs weight, but with very different magnitudes:

- **`elevation`** achieves the highest Hausdorff distance (~50 px at peak), meaning it produces the most divergent paths from the straight-line baseline. This makes sense: the `elevation` model penalizes elevation *differences* between successive cells, so it actively routes around ridges and valleys.
- **`energy`** and **`slope_uphill`** reach intermediate peaks (~40 px). The `energy` model penalizes cumulative elevation gain, while `slope_uphill` only penalizes uphill slopes — both produce meaningful but more moderate deviations.
- **`absolute_elevation`** has the lowest Hausdorff distance (~5 px). This model penalizes *absolute* elevation values rather than gradients, so in crops with relatively uniform elevation, it produces near-straight paths. It only deviates when there are large absolute elevation differences (e.g., sea-level vs mountains).

**Per-model subplots (with peaks):** The red markers show the weight at which each model's Hausdorff distance peaks. Beyond these peaks, further weight increases yield diminishing returns (the path has already deviated as far as the 256×256 crop allows). The ±1σ bands show significant sample-to-sample variance, especially for `elevation`, reflecting the diversity of terrain encountered across 1000 random crops.

> **Key insight:** Raw Hausdorff maximization alone would always pick the highest possible weight. But as we'll see in the boundary analysis, very high weights cause paths to hug image edges — an artifact, not a meaningful route.

---

## 4. Boundary Hit Analysis

At high weights, paths may deviate so much that they hug image boundaries —
an artifact of the finite crop, not a meaningful route choice.

In [ ]:
# Boundary hit rate and boundary fraction vs weight
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: boundary hit rate (fraction of samples with any boundary contact)
ax = axes[0]
for cm in COST_MODELS_TUNABLE:
    sub = agg_df[agg_df["cost_model"] == cm].sort_values("weight")
    ax.plot(sub["weight"], sub["boundary_hit_rate"], "o-", lw=1.2, markersize=3, label=cm, color=CM_COLORS[cm])
ax.set_xscale("log")
ax.set_xlabel("Weight")
ax.set_ylabel("Boundary hit rate (fraction of samples)")
ax.set_title("Boundary Hit Rate vs Weight")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8)
ax.grid(True, ls=":", alpha=0.5)

# Right: mean boundary fraction (fraction of path points at boundary)
ax = axes[1]
for cm in COST_MODELS_TUNABLE:
    sub = agg_df[agg_df["cost_model"] == cm].sort_values("weight")
    ax.plot(sub["weight"], sub["mean_boundary_frac"], "s-", lw=1.2, markersize=3, label=cm, color=CM_COLORS[cm])
ax.set_xscale("log")
ax.set_xlabel("Weight")
ax.set_ylabel("Mean boundary fraction")
ax.set_title("Mean Boundary Fraction vs Weight")
ax.legend(fontsize=8)
ax.grid(True, ls=":", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Per-cost-model boundary fraction subplots
fig, axes = plt.subplots(1, n_cm, figsize=(5 * n_cm, 4), sharey=True)
if n_cm == 1:
    axes = [axes]

for idx, cm in enumerate(COST_MODELS_TUNABLE):
    ax = axes[idx]
    sub = agg_df[agg_df["cost_model"] == cm].sort_values("weight")
    ax.fill_between(sub["weight"], 0, sub["mean_boundary_frac"], alpha=0.3, color=CM_COLORS[cm])
    ax.plot(sub["weight"], sub["mean_boundary_frac"], "s-", lw=1.2, markersize=3, color=CM_COLORS[cm])
    ax.set_xscale("log")
    ax.set_title(cm, fontsize=11)
    ax.set_xlabel("Weight")
    if idx == 0:
        ax.set_ylabel("Mean boundary fraction")
    ax.grid(True, ls=":", alpha=0.5)

fig.suptitle("Boundary Fraction per Cost Model", fontsize=13)
plt.tight_layout()
plt.show()

### Analysis: Boundary Artifacts

**Left panel — Boundary hit rate:** The fraction of samples where *any* path point falls within 2 pixels of the crop edge. For most models this remains near zero for moderate weights, but:
- **`elevation`** stands out with a boundary hit rate climbing to ~20-40% at high weights ($w > 10^4$). This is the most aggressive model — when terrain penalties dominate, the solver finds it cheaper to route along the flat boundary pixels (which have constant elevation) than to cross real terrain features.
- **`slope_uphill`** shows a smaller but non-negligible boundary hit rate at very high weights.
- **`absolute_elevation`** and **`energy`** remain well-behaved with near-zero boundary contact across all weights.

**Right panel — Mean boundary fraction:** Even among samples that *do* hit the boundary, how much of the path is affected? The `elevation` model is again the outlier, with up to ~4% of path points near the boundary at extreme weights. This confirms the need for a penalty term.

**Per-model area charts:** Visualize the boundary fraction profile per model. The `elevation` model's boundary fraction rises sharply above $w \approx 10^3$, precisely the weight range where Hausdorff distance also saturates — confirming that the additional "deviation" is actually boundary hugging, not meaningful terrain avoidance.

> **Rationale for the boundary penalty:** Without penalizing boundary contact, weight tuning would select extreme weights that produce visually implausible paths running along crop edges. The term $(1 - B(w))$ in the combined score elegantly suppresses these artifact weights.

---

## 5. Weight Tuning: Combined Score

Compute the combined score $\text{score}(w) = \hat{H}(w) \cdot (1 - B(w)) \cdot P_{\text{run}}(w)$
and select the optimal weight per cost model.

In [ ]:
# Compute normalized Hausdorff, contiguous-run penalty, and combined score
tune_df = agg_df[agg_df["cost_model"] != "distance"].copy()

tune_df["h_norm"] = tune_df.groupby("cost_model")["mean_hausdorff"].transform(
    lambda s: s / s.max() if s.max() > 0 else 0.0
)

# Contiguous-run penalty: linear ramp from 1→0 between threshold and 2×threshold
tune_df["run_penalty"] = 1.0 - np.clip(
    (tune_df["mean_max_contig_boundary"] - MAX_BOUNDARY_RUN_FRAC) / MAX_BOUNDARY_RUN_FRAC,
    0.0,
    1.0,
)

tune_df["score"] = tune_df["h_norm"] * (1.0 - tune_df["mean_boundary_frac"]) * tune_df["run_penalty"]
tune_df.sort_values(["cost_model", "weight"], inplace=True)

tune_df[
    ["cost_model", "weight", "h_norm", "mean_boundary_frac", "mean_max_contig_boundary", "run_penalty", "score"]
].head(10)

In [ ]:
# Select optimal weight per cost model
weights_tuned = {}
tuning_details = {}

for cm in COST_MODELS_TUNABLE:
    cm_df = tune_df[tune_df["cost_model"] == cm].sort_values("weight")
    w_arr = cm_df["weight"].values
    scores = cm_df["score"].values
    hausdorffs = cm_df["mean_hausdorff"].values
    b_fracs = cm_df["mean_boundary_frac"].values
    contig_runs = cm_df["mean_max_contig_boundary"].values
    run_pens = cm_df["run_penalty"].values

    max_score = scores.max()
    if max_score <= 0:
        weights_tuned[cm] = 1.0
        tuning_details[cm] = {"max_score": 0, "chosen_idx": 0}
        continue

    # Among weights achieving >= SCORE_THRESHOLD of max, pick smallest
    candidates = np.where(scores >= SCORE_THRESHOLD * max_score)[0]
    idx = int(candidates[0]) if len(candidates) > 0 else int(np.argmax(scores))

    weights_tuned[cm] = float(w_arr[idx])
    tuning_details[cm] = {
        "max_score": float(max_score),
        "chosen_score": float(scores[idx]),
        "chosen_hausdorff": float(hausdorffs[idx]),
        "chosen_boundary_frac": float(b_fracs[idx]),
        "chosen_max_contig_run": float(contig_runs[idx]),
        "chosen_run_penalty": float(run_pens[idx]),
        "n_candidates": len(candidates),
    }

print("Tuned weights (Hausdorff + boundary + contig-run penalty + asymptote minimization):")
for cm, w in sorted(weights_tuned.items()):
    d = tuning_details[cm]
    print(
        f"  {cm:25s} -> w={w:12.1f}  "
        f"(score={d.get('chosen_score', 0):.4f}, "
        f"H={d.get('chosen_hausdorff', 0):.2f}, "
        f"B={d.get('chosen_boundary_frac', 0):.4f}, "
        f"C={d.get('chosen_max_contig_run', 0):.4f}, "
        f"P_run={d.get('chosen_run_penalty', 1):.3f})"
    )

In [ ]:
# 4-panel per-cost-model: Hausdorff | Boundary | Contig Run Penalty | Combined score
fig, axes = plt.subplots(n_cm, 4, figsize=(24, 5 * n_cm), squeeze=False)
fig.suptitle(
    f"Weight Tuning (score threshold = {SCORE_THRESHOLD:.0%}, max contig run = {MAX_BOUNDARY_RUN_FRAC:.0%})",
    fontsize=14,
    y=1.01,
)

for row, cm in enumerate(COST_MODELS_TUNABLE):
    cm_df = tune_df[tune_df["cost_model"] == cm].sort_values("weight")
    ws = cm_df["weight"].values
    hs = cm_df["mean_hausdorff"].values
    bs = cm_df["mean_boundary_frac"].values
    cs = cm_df["mean_max_contig_boundary"].values
    rps = cm_df["run_penalty"].values
    ss = cm_df["score"].values
    w_opt = weights_tuned[cm]

    # Col 0: Hausdorff
    ax = axes[row, 0]
    ax.plot(ws, hs, "o-", lw=1.5, color="tab:blue")
    ax.axvline(w_opt, color="red", ls="--", lw=1, label=f"w*={w_opt:.0f}")
    ax.set_xscale("log")
    ax.set_ylabel("Mean Hausdorff (px)")
    ax.set_title(f"{cm} — Hausdorff")
    ax.legend(fontsize=8)
    ax.grid(True, ls=":", alpha=0.5)

    # Col 1: Boundary fraction
    ax = axes[row, 1]
    ax.plot(ws, bs, "s-", lw=1.5, color="tab:orange")
    ax.axvline(w_opt, color="red", ls="--", lw=1, label=f"w*={w_opt:.0f}")
    ax.set_xscale("log")
    ax.set_ylabel("Mean boundary fraction")
    ax.set_title(f"{cm} — Boundary frac")
    ax.legend(fontsize=8)
    ax.grid(True, ls=":", alpha=0.5)

    # Col 2: Contiguous run & penalty
    ax = axes[row, 2]
    ax.plot(ws, cs, "^-", lw=1.5, color="tab:purple", label="Mean max contig run")
    ax.axhline(MAX_BOUNDARY_RUN_FRAC, color="gray", ls=":", lw=1, label=f"θ={MAX_BOUNDARY_RUN_FRAC:.0%}")
    ax.axhline(
        2 * MAX_BOUNDARY_RUN_FRAC, color="gray", ls="--", lw=1, alpha=0.5, label=f"2θ={2 * MAX_BOUNDARY_RUN_FRAC:.0%}"
    )
    ax2 = ax.twinx()
    ax2.plot(ws, rps, "v-", lw=1, color="tab:red", alpha=0.6, label="P_run")
    ax2.set_ylabel("Run penalty P_run", color="tab:red", fontsize=8)
    ax2.set_ylim(-0.05, 1.15)
    ax.axvline(w_opt, color="red", ls="--", lw=1, label=f"w*={w_opt:.0f}")
    ax.set_xscale("log")
    ax.set_ylabel("Max contiguous boundary frac")
    ax.set_title(f"{cm} — Contig Run")
    # Merge legends from both axes
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left")
    ax.grid(True, ls=":", alpha=0.5)

    # Col 3: Combined score
    ax = axes[row, 3]
    ax.plot(ws, ss, "D-", lw=1.5, color="tab:green")
    max_s = ss.max()
    ax.axhline(SCORE_THRESHOLD * max_s, color="purple", ls=":", lw=1, label=f"{SCORE_THRESHOLD:.0%} threshold")
    ax.axvline(w_opt, color="red", ls="--", lw=1, label=f"w*={w_opt:.0f}")
    opt_idx = int(np.searchsorted(ws, w_opt))
    ax.scatter([w_opt], [ss[min(opt_idx, len(ss) - 1)]], color="red", s=60, zorder=10)
    ax.set_xscale("log")
    ax.set_ylabel("Score = Ĥ·(1-B)·P_run")
    ax.set_title(f"{cm} — Score")
    ax.legend(fontsize=8)
    ax.grid(True, ls=":", alpha=0.5)

    if row == n_cm - 1:
        for c in range(4):
            axes[row, c].set_xlabel("Weight")

plt.tight_layout()
plt.show()

In [ ]:
# All cost models overlaid: combined score vs weight
fig, ax = plt.subplots(figsize=(12, 6))

for cm in COST_MODELS_TUNABLE:
    cm_df = tune_df[tune_df["cost_model"] == cm].sort_values("weight")
    ax.plot(
        cm_df["weight"],
        cm_df["score"],
        "o-",
        lw=1.5,
        markersize=4,
        label=cm,
        color=CM_COLORS[cm],
    )
    # Mark chosen weight
    w_opt = weights_tuned[cm]
    opt_row = cm_df.iloc[(cm_df["weight"] - w_opt).abs().argsort().iloc[0]]
    ax.scatter([w_opt], [opt_row["score"]], color="red", s=60, zorder=10, edgecolors="black", linewidths=0.5)
    ax.annotate(
        f"{w_opt:.0f}",
        xy=(w_opt, opt_row["score"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8,
        color="red",
    )

ax.set_xscale("log")
ax.set_xlabel("Weight")
ax.set_ylabel("Combined score")
ax.set_title("Combined Score vs Weight (all tunable cost models)")
ax.legend(fontsize=9)
ax.grid(True, ls=":", alpha=0.5)
plt.tight_layout()
plt.show()

### Analysis: Weight Tuning Results

**4-panel per-model view (Hausdorff | Boundary | Contig Run | Score):**
Each row shows the four signals for one cost model. The red dashed line marks the selected weight $w^*$:

- **Column 1 — Hausdorff:** Path diversity from the distance baseline (higher = more diverse).
- **Column 2 — Boundary fraction:** Overall fraction of path points near the edge.
- **Column 3 — Contiguous run penalty:** The purple curve shows the mean longest contiguous boundary run as a fraction of path length. The gray dotted/dashed lines mark the $\theta$ (3%) and $2\theta$ (6%) thresholds. The red overlay shows the resulting penalty $P_{\text{run}}$: it equals 1.0 below $\theta$, ramps linearly to 0.0 at $2\theta$, and stays at 0.0 beyond. This penalizes weights where paths run along edges for extended stretches, especially affecting the `elevation` cost model.
- **Column 4 — Combined score:** The product $\hat{H} \cdot (1-B) \cdot P_{\text{run}}$.

With $\theta = 3\%$, `elevation` now selects $w^* = 62.4$ — a compromise between the overly strict 1% threshold ($w^* = 11.9$) and the permissive 10% threshold ($w^* = 142.5$). At this weight, the elevation model achieves a Hausdorff distance of ~48 px with a contiguous run of ~2.6%, staying within the 3% bound. The score plateaus around 0.88, reflecting a good balance between path diversity and boundary control.

> **Asymptote minimization principle:** Among all weights achieving $\ge 95\%$ of the peak score, we always pick the *smallest*. This avoids unnecessarily large weights that would waste computation on negligible improvements and risk overfitting the weight to specific terrain patterns.

---

## 6. Summary

In [ ]:
# Summary table
summary_rows = []
for cm in sorted(weights_tuned.keys()):
    d = tuning_details[cm]
    summary_rows.append(
        {
            "cost_model": cm,
            "weight": weights_tuned[cm],
            "score": d.get("chosen_score", 0),
            "hausdorff": d.get("chosen_hausdorff", 0),
            "boundary_frac": d.get("chosen_boundary_frac", 0),
            "max_contig_run": d.get("chosen_max_contig_run", 0),
            "run_penalty": d.get("chosen_run_penalty", 1),
        }
    )

summary_table = pd.DataFrame(summary_rows).set_index("cost_model")
display(summary_table)

print("\nFinal tuned weights:")
for cm, w in sorted(weights_tuned.items()):
    print(f"  {cm}: {w:.1f}")

### Summary Interpretation

The summary table shows the final tuned weights with both boundary metrics:

- **`boundary_frac`**: overall fraction of path points near edges (global average)
- **`max_contig_run`**: longest contiguous boundary segment as fraction of total path
- **`run_penalty`**: the multiplicative penalty applied from the contiguous-run constraint ($P_{\text{run}}$)

With $\theta = 3\%$, all tuned weights produce paths with contiguous runs below the threshold ($P_{\text{run}} = 1.0$ for all). The `elevation` model selects $w^* = 62.4$ with a contiguous run of 2.6% — just under the 3% limit — while achieving a Hausdorff distance of ~48 px. The other three models remain unaffected since their contiguous runs are well below 1%.

> **Practical implication:** These tuned weights can be used directly in `passage generate` to produce a balanced dataset of shortest-path annotations. Each cost model at its tuned weight creates meaningfully different paths from the distance baseline while keeping boundary artifacts — both overall and contiguous — under control.

---

## 7. Final Visualization with Tuned Weights

Show paths from all non-distance cost models at their tuned weight,
side by side on fresh samples.

In [ ]:
# Generate display samples with tuned weights
t0 = time.time()
ctx = mp.get_context("fork")
with ctx.Pool(NUM_WORKERS) as pool:
    results = list(
        tqdm(
            pool.imap_unordered(_process_one_display_sample, range(NUM_DISPLAY_SAMPLES)),
            total=NUM_DISPLAY_SAMPLES,
            desc=f"Display samples ({NUM_WORKERS} workers)",
        )
    )

display_samples = sorted([r for r in results if r is not None], key=lambda s: s["index"])

elapsed = time.time() - t0
print(f"Display samples: {len(display_samples)} in {elapsed:.1f}s ({NUM_WORKERS} workers)")

In [ ]:
# Visualize each display sample
display_cms = sorted(weights_tuned.keys())
n_cols = len(display_cms)

legend_handles = [
    Line2D([0], [0], color="red", lw=2, label="Tuned path"),
    Line2D([0], [0], color="gray", lw=1, ls="--", label="Distance baseline"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="lime", markersize=6, label="Start"),
    Line2D([0], [0], marker="x", color="cyan", markersize=6, label="End"),
]

for sample in display_samples:
    crop = sample["crop"]
    start = sample["start"]
    end = sample["end"]
    paths = sample["paths"]
    elev_min, elev_max = float(np.min(crop)), float(np.max(crop))

    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    if n_cols == 1:
        axes = [axes]

    for col, cm in enumerate(display_cms):
        ax = axes[col]
        ax.imshow(crop, cmap="terrain", vmin=elev_min, vmax=elev_max, origin="upper")

        # Distance baseline in gray
        dp = paths.get("distance", [])
        if len(dp) > 0:
            pa = np.array(dp)
            ax.plot(pa[:, 1], pa[:, 0], color="gray", lw=1, ls="--", alpha=0.6)

        # Tuned path in red
        path = paths[cm]
        if len(path) > 0:
            pa = np.array(path)
            ax.plot(pa[:, 1], pa[:, 0], color="red", lw=2)

        ax.plot(start[1], start[0], "o", color="lime", markersize=6)
        ax.plot(end[1], end[0], "x", color="cyan", markersize=6)
        w = weights_tuned[cm]
        ax.set_title(f"{cm}  (w={w:.0f})", fontsize=10)
        ax.axis("off")

    fig.legend(handles=legend_handles, loc="upper right", fontsize=8)
    fig.suptitle(f"Sample {sample['index']}  |  elev=[{elev_min:.0f}, {elev_max:.0f}]", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# All tuned paths overlaid on a single elevation map per sample
for sample in display_samples:
    crop = sample["crop"]
    start = sample["start"]
    end = sample["end"]
    paths = sample["paths"]
    elev_min, elev_max = float(np.min(crop)), float(np.max(crop))

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(crop, cmap="terrain", vmin=elev_min, vmax=elev_max, origin="upper")

    for cm in display_cms:
        path = paths[cm]
        if len(path) > 0:
            pa = np.array(path)
            ax.plot(pa[:, 1], pa[:, 0], lw=2, label=f"{cm} (w={weights_tuned[cm]:.0f})", color=CM_COLORS[cm])

    # Distance baseline
    dp = paths.get("distance", [])
    if len(dp) > 0:
        pa = np.array(dp)
        ax.plot(pa[:, 1], pa[:, 0], color="gray", lw=1.5, ls="--", label="distance", alpha=0.7)

    ax.plot(start[1], start[0], "o", color="lime", markersize=8, zorder=10)
    ax.plot(end[1], end[0], "x", color="cyan", markersize=8, zorder=10)
    ax.legend(fontsize=8, loc="lower left")
    ax.set_title(f"Sample {sample['index']}  |  elev=[{elev_min:.0f}, {elev_max:.0f}]", fontsize=11)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

### Analysis: Visual Validation of Tuned Weights

**Side-by-side view** (one subplot per cost model): Each panel shows the tuned path (red) against the distance baseline (gray dashed). Key observations:

- **`absolute_elevation`:** Paths are nearly identical to the distance baseline in most samples, consistent with its low Hausdorff distance (~5 px). This model is only useful for subtle variation.
- **`elevation`:** Produces the most visually distinct paths — long detours that follow contour lines, avoiding steep gradients. In some samples (e.g., mountainous terrain), the path may traverse the entire crop perimeter to stay on gentle slopes.
- **`energy`** and **`slope_uphill`:** Intermediate behavior — paths visibly avoid steep terrain but remain more direct than `elevation`. The difference between these two models is subtle and depends on whether the terrain has symmetric or asymmetric slopes.

**Overlay view** (all models on one map): When multiple cost models are superimposed on the same elevation map, we can directly compare their routing strategies:
- On flat terrain (e.g., Sample 7 with elev=[0, 0]), all models produce identical straight-line paths — correct behavior since there is no terrain to avoid.
- On rugged terrain, the paths fan out, with `elevation` taking the widest detour and `absolute_elevation` staying closest to the diagonal.
- Samples near coastlines or tile edges (with zero-elevation ocean pixels) expose boundary-hugging behavior that our tuning procedure successfully mitigates — at the tuned weights, paths may approach but do not systematically follow the boundary.

> **Validation:** The visual results confirm that the automated weight selection produces qualitatively sensible paths across diverse terrain conditions, and that none of the tuned weights lead to degenerate boundary-following behavior.

In [ ]:
weights_tuned